
**Making pdf RAG**

In [ ]:
!pip install -q langchain langchain-community

!pip install -U langchain-google-genai

In [ ]:
!pip install -qU langchain-huggingface langchain-core langchain-chroma chromadb langchain.retrievers
!pip install -U opentelemetry-sdk opentelemetry-api opentelemetry-exporter-otlp sentence-transformers

In [ ]:
!pip install pypdf langchain-chroma
!pip show langchain-classic langchain-community
!pip install -q -U langchain langchain-community rank_bm25
!pip install -q rank_bm25 langchain-community

**Api key**

In [ ]:
from google.colab import userdata
import os

api_key = userdata.get('GOOGLE_API_KEY')
os.environ['GOOGLE_API_KEY'] = api_key

# All imports

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import  RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_chroma import Chroma
from sentence_transformers import CrossEncoder
llm=ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

/tmp/ipykernel_2497/3542462758.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


**Loading PDF**

In [ ]:
loader=PyPDFLoader("/content/AI_Engineer_Roadmap_v3.md.pdf")
docs=loader.load()
print("Total number of pages are in you documnets =" ,len(docs))

Total number of pages are in you documnets = 22


**Making chunks**

In [ ]:
text=RecursiveCharacterTextSplitter(chunk_size=1200,chunk_overlap=300)
chunks=text.split_documents(docs)
print("Total number of chunks =" ,len(chunks))
print(chunks[1].page_content)


Total number of chunks = 46
LLM/RAG/backend/deployment layer needed to genuinely apply for "AI Engineer" titles, not
just "Junior ML Engineer."
Week 1: Math Intuition + ML Foundations (Aug 11–17)
Skipped: Python/pandas/numpy/EDA basics — you already have this.
Math (3-4 days):
3Blue1Brown — Essence of Linear Algebra (playlist)
3Blue1Brown — Essence of Calculus (playlist)
StatQuest — full channel & video index | StatQuest YouTube
Intro ML theory (3 days):
Andrew Ng — Machine Learning Specialization (Coursera, audit free) — beginner-friendly,
practical; use this as your primary source, not the old grad-level CS229 lectures
Kaggle Learn — Intro to Machine Learning (free micro-course) — a few hours, hands-on, great
since you already know pandas/EDA
Primary channel pick: CampusX — 100 Days of Machine Learning — CONFIRMED primary
source (Hindi explanations are working well for retention; stick with this throughout)
Day 7: Buffer/rest + light review


# ** Embedding and Vector Store **


In [ ]:
Embedding = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vectorstore = Chroma.from_documents(documents=chunks, embedding=Embedding ,persist_directory="./chroma_Roa_db")

In [ ]:
print(vectorstore._collection.count())

46


**Hybrind retriever**

In [ ]:
bm25=BM25Retriever.from_documents(chunks)
bm25.k=5


retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

hybrid_retriever = EnsembleRetriever(retrievers=[bm25, retriever],weights=[0.4,0.6])

Re-**Ranking::\\**

In [ ]:
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [ ]:
query = "What is explicitly out of scope for the cloud deployment phase??"  # ask something specific to your actual PDF content
results = hybrid_retriever.invoke(query)

scored=[]
for doc in results:
    score =reranker.predict([[query, doc.page_content]])[0]
    scored.append((score, doc))

scored.sort(key=lambda x :x[0] ,reverse=True)

best=scored[:3]

for score, doc in best:
    print("Score:", score)
    print("Page:", doc.metadata.get("page", "?"))
    print(doc.page_content[:150])
    print("-----")

Score: 4.6764297
Page: 16
Usage limits and cloud/API billing alerts — "free tier" is not a guarantee of zero charges, and
alerts alone don't cap spending; set actual limits whe
-----
Score: -2.8413048
Page: 17
real evidence comparison, light RNN/LSTM overview)
Week 5: ✅  Complete — and then some. Churn ANN deployed as a real product: clean module
structure, 
-----
Score: -4.7330737
Page: 15
Resources:
Docker — Ofﬁcial Get Started Guide
Docker Docs — FastAPI containerization samples
Project: Containerize your FastAPI + RAG project from Wee
-----


In [ ]:
def rerank(query, docs, top_k=3):

    scored = []
    for doc in docs:
        score = reranker.predict([[query, doc.page_content]])[0]
        scored.append((score, doc))

    scored.sort(key=lambda x: x[0], reverse=True)

    best_docs = []
    for score, doc in scored[:top_k]:
        best_docs.append(doc)
    return best_docs

# LLM

In [ ]:

def ask_pdf(query):
    candidates = hybrid_retriever.invoke(query)
    best_docs = rerank(query, candidates, top_k=3)

    context = ""
    for doc in best_docs:
        context = context + "[Page " + str(doc.metadata.get("page", "?")) + "]: " + doc.page_content + "\n\n"

    prompt = f"""Answer the question using only the context below.
If the context contains information that answers the question, give that answer, even if the wording of the question is slightly different from the text.
Say "I don't know" only if the context has no relevant information at all.

Context:
{context}

Question: {query}

Answer:"""

    response = llm.invoke(prompt)
    if isinstance(response.content, list):
        answer = response.content[0]["text"]
    else:
        answer = response.content
    return answer

In [ ]:
while True:
  query=input("You ")
  if query=="by":
    print("by by....")
    break
  print(ask_pdf(query))


Based on the provided context, the text mentions that Kubernetes, distributed training, complex multi-agent systems, and extensive fine-tuning are explicitly out of scope for "this phase" (Page 16). However, the context does not mention a cloud deployment phase specifically being associated with these exclusions, nor does it state what is explicitly out of scope for a cloud deployment phase. 

I don't know.
